# Advanced Feature Engineering Pipeline

This notebook demonstrates a comprehensive automated feature engineering system with:
- Automated feature creation
- Advanced transformations
- Time series feature extraction
- Feature selection strategies
- End-to-end pipelines

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Feature engineering imports
from src.feature_engineering.automated_feature_creator import AutomatedFeatureEngineer
from src.feature_engineering.feature_transformers import (
    OutlierTransformer, SkewnessTransformer, DimensionalityReducer,
    ClusteringTransformer, InteractionTransformer, BinningTransformer,
    TargetTransformer, FeatureAugmenter, create_transformation_pipeline
)
from src.feature_engineering.time_series_features import (
    TimeSeriesFeatureExtractor, SeasonalFeatureExtractor,
    WindowFeatureExtractor, ChangePointDetector
)
from src.feature_engineering.feature_selection import AdvancedFeatureSelector, FeatureImportanceAnalyzer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Feature Engineering Pipeline Ready!")

## 1. Generate Sample Data

In [ ]:
# Classification dataset
X_class, y_class = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=2,
    random_state=42
)

# Convert to DataFrame
feature_names = [f'feature_{i}' for i in range(20)]
df_class = pd.DataFrame(X_class, columns=feature_names)
df_class['target'] = y_class

print(f"Classification dataset shape: {df_class.shape}")
print(f"Class distribution: {y_class.mean():.2%} positive")

# Regression dataset
X_reg, y_reg = make_regression(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    noise=0.1,
    random_state=42
)

df_reg = pd.DataFrame(X_reg, columns=feature_names)
df_reg['target'] = y_reg

print(f"\nRegression dataset shape: {df_reg.shape}")
print(f"Target range: [{y_reg.min():.2f}, {y_reg.max():.2f}]")

In [ ]:
# Create time series dataset
np.random.seed(42)
n_samples = 1000
time_index = pd.date_range('2023-01-01', periods=n_samples, freq='H')

# Generate time series with trend, seasonality, and noise
trend = np.linspace(100, 150, n_samples)
seasonal = 10 * np.sin(2 * np.pi * np.arange(n_samples) / 24)  # Daily pattern
noise = np.random.normal(0, 5, n_samples)

df_ts = pd.DataFrame({
    'timestamp': time_index,
    'value': trend + seasonal + noise,
    'feature_1': np.random.randn(n_samples).cumsum(),
    'feature_2': np.random.randn(n_samples) * 10,
    'entity_id': np.random.choice(['A', 'B', 'C'], n_samples)
})

print(f"Time series dataset shape: {df_ts.shape}")
df_ts.head()

## 2. Automated Feature Engineering

In [ ]:
# Initialize automated feature engineer
feature_engineer = AutomatedFeatureEngineer(
    task_type='classification',
    max_features=50,
    feature_selection_method='mutual_info',
    verbosity=1
)

# Split data
X = df_class.drop('target', axis=1)
y = df_class['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Engineer features
X_train_engineered = feature_engineer.engineer_features(
    X_train,
    target=y_train,
    include_interactions=True,
    include_polynomial=True,
    include_aggregates=False,
    include_deep_features=False
)

print(f"\nOriginal features: {X_train.shape[1]}")
print(f"Engineered features: {X_train_engineered.shape[1]}")

# Transform test data
X_test_engineered = feature_engineer.transform(X_test)

# Get feature importance
importance = feature_engineer.get_feature_importance()
print(f"\nTop 10 important features:")
for feat, score in list(importance.items())[:10]:
    print(f"  {feat}: {score:.4f}")

In [ ]:
# Visualize feature importance
if importance:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    top_features = list(importance.keys())[:20]
    top_scores = list(importance.values())[:20]
    
    ax.barh(range(len(top_features)), top_scores)
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features)
    ax.set_xlabel('Importance Score')
    ax.set_title('Top 20 Feature Importances')
    
    plt.tight_layout()
    plt.show()

## 3. Feature Transformation Pipeline

In [ ]:
# Create transformation pipeline
transformers = [
    OutlierTransformer(method='iqr', threshold=1.5),
    SkewnessTransformer(threshold=0.5, method='boxcox'),
    InteractionTransformer(interaction_type='multiply', max_features=5),
    ClusteringTransformer(method='kmeans', n_clusters=5),
    BinningTransformer(strategy='quantile', n_bins=5, encode='ordinal')
]

# Apply transformations
X_transformed = X_train.copy()

for transformer in transformers:
    print(f"Applying {transformer.__class__.__name__}...")
    transformer.fit(X_transformed)
    X_transformed = transformer.transform(X_transformed)
    print(f"  Shape after transformation: {X_transformed.shape}")

print(f"\nFinal shape: {X_transformed.shape}")

In [ ]:
# Dimensionality reduction comparison
methods = ['pca', 'svd', 'ica', 'factor']
n_components = 10

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, method in enumerate(methods):
    reducer = DimensionalityReducer(method=method, n_components=n_components)
    
    try:
        reducer.fit(X_train)
        X_reduced = reducer.transform(X_train)
        
        # Plot first two components
        if X_reduced.shape[1] >= 2:
            axes[idx].scatter(X_reduced[:, 0], X_reduced[:, 1], c=y_train, cmap='viridis', alpha=0.6)
            axes[idx].set_title(f'{method.upper()} - First 2 Components')
            axes[idx].set_xlabel('Component 1')
            axes[idx].set_ylabel('Component 2')
            
            # Add explained variance if available
            if hasattr(reducer.reducer, 'explained_variance_ratio_'):
                var_explained = reducer.reducer.explained_variance_ratio_[:2].sum()
                axes[idx].text(0.02, 0.98, f'Var explained: {var_explained:.2%}',
                             transform=axes[idx].transAxes, verticalalignment='top')
    except Exception as e:
        axes[idx].text(0.5, 0.5, f'{method} failed', ha='center', va='center')
        axes[idx].set_title(f'{method.upper()}')

plt.tight_layout()
plt.show()

## 4. Time Series Feature Extraction

In [ ]:
# Initialize time series feature extractor
ts_extractor = TimeSeriesFeatureExtractor(
    window_sizes=[5, 10, 20],
    include_statistical=True,
    include_frequency=True,
    include_entropy=True,
    include_autocorrelation=True,
    include_tsfresh=False
)

# Extract features
ts_features = ts_extractor.extract_features(
    df_ts,
    time_col='timestamp',
    value_cols=['value', 'feature_1'],
    entity_col='entity_id'
)

print(f"Time series features shape: {ts_features.shape}")
print(f"\nSample features:")
print(ts_features.columns[:20].tolist())

In [ ]:
# Apply seasonal decomposition
seasonal_extractor = SeasonalFeatureExtractor(period=24, method='fourier')
seasonal_features = seasonal_extractor.transform(df_ts[['value', 'feature_1']])

# Visualize seasonal patterns
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Original series
axes[0, 0].plot(df_ts.index[:100], df_ts['value'].iloc[:100])
axes[0, 0].set_title('Original Time Series')
axes[0, 0].set_xlabel('Time')

# Fourier components
fourier_cols = [col for col in seasonal_features.columns if 'sin' in col or 'cos' in col]
if fourier_cols:
    for col in fourier_cols[:3]:
        axes[0, 1].plot(seasonal_features.index[:100], seasonal_features[col].iloc[:100], label=col, alpha=0.7)
    axes[0, 1].set_title('Fourier Components')
    axes[0, 1].legend()

# Window features
window_extractor = WindowFeatureExtractor(window_size=24, step_size=1)
window_features = window_extractor.transform(df_ts[['value']])

axes[1, 0].plot(window_features.index[:100], window_features['value_window_mean'].iloc[:100], label='Mean')
axes[1, 0].plot(window_features.index[:100], window_features['value_window_std'].iloc[:100], label='Std')
axes[1, 0].set_title('Rolling Window Features')
axes[1, 0].legend()

# Change point detection
change_detector = ChangePointDetector(method='cusum', threshold=0.05)
change_detector.fit(df_ts[['value']])
change_features = change_detector.transform(df_ts[['value']])

axes[1, 1].plot(df_ts.index[:200], df_ts['value'].iloc[:200])
change_points = df_ts.index[change_features['value_is_changepoint'] == 1][:200]
for cp in change_points:
    axes[1, 1].axvline(x=cp, color='red', linestyle='--', alpha=0.5)
axes[1, 1].set_title('Change Point Detection')

plt.tight_layout()
plt.show()

## 5. Advanced Feature Selection

In [ ]:
# Initialize advanced feature selector
selector = AdvancedFeatureSelector(
    task_type='classification',
    selection_methods=['variance', 'correlation', 'mutual_info', 'importance', 'lasso'],
    max_features=15,
    cv_folds=5,
    verbosity=1
)

# Fit and select features
X_selected, feature_scores = selector.fit_select(
    X_train_engineered,
    y_train,
    return_scores=True
)

print(f"\nSelected features: {X_selected.shape[1]}")
print(f"Selected feature names: {X_selected.columns.tolist()[:10]}...")

In [ ]:
# Compare selection methods
selection_methods = ['mutual_info', 'importance', 'lasso', 'rfe', 'forward']
results = {}

for method in selection_methods:
    print(f"\nTesting {method} selection...")
    
    selector = AdvancedFeatureSelector(
        task_type='classification',
        selection_methods=[method],
        max_features=10,
        cv_folds=3,
        verbosity=0
    )
    
    try:
        X_selected = selector.fit_select(X_train, y_train)
        
        # Train model on selected features
        model = RandomForestClassifier(n_estimators=50, random_state=42)
        model.fit(X_selected, y_train)
        
        # Transform test set
        X_test_selected = X_test[X_selected.columns]
        
        # Evaluate
        train_score = model.score(X_selected, y_train)
        test_score = model.score(X_test_selected, y_test)
        
        results[method] = {
            'n_features': X_selected.shape[1],
            'train_score': train_score,
            'test_score': test_score,
            'features': X_selected.columns.tolist()
        }
        
        print(f"  Features: {X_selected.shape[1]}, Train: {train_score:.3f}, Test: {test_score:.3f}")
        
    except Exception as e:
        print(f"  Method {method} failed: {e}")
        results[method] = {'n_features': 0, 'train_score': 0, 'test_score': 0}

In [ ]:
# Visualize selection results
if results:
    methods = list(results.keys())
    train_scores = [results[m]['train_score'] for m in methods]
    test_scores = [results[m]['test_score'] for m in methods]
    n_features = [results[m]['n_features'] for m in methods]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Score comparison
    x = np.arange(len(methods))
    width = 0.35
    
    axes[0].bar(x - width/2, train_scores, width, label='Train', alpha=0.8)
    axes[0].bar(x + width/2, test_scores, width, label='Test', alpha=0.8)
    axes[0].set_xlabel('Selection Method')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Performance by Selection Method')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(methods, rotation=45)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Features vs performance
    axes[1].scatter(n_features, train_scores, label='Train', s=100, alpha=0.7)
    axes[1].scatter(n_features, test_scores, label='Test', s=100, alpha=0.7)
    
    for i, method in enumerate(methods):
        axes[1].annotate(method, (n_features[i], test_scores[i]), 
                        textcoords="offset points", xytext=(0,5), ha='center')
    
    axes[1].set_xlabel('Number of Features')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Features vs Performance')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Analyze feature importance with multiple models
importance_analyzer = FeatureImportanceAnalyzer()

importance_df = importance_analyzer.analyze(
    X_train,
    y_train,
    models=['rf', 'gb', 'et', 'lr']
)

print("Feature Importance Analysis:")
print(importance_df.head(10))

In [ ]:
# Visualize importance across models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

models = ['rf', 'gb', 'et', 'lr']
model_names = ['Random Forest', 'Gradient Boosting', 'Extra Trees', 'Linear Model']

for idx, (model, name) in enumerate(zip(models, model_names)):
    if model in importance_df.columns:
        top_features = importance_df.nlargest(10, model)
        
        axes[idx].barh(range(len(top_features)), top_features[model].values)
        axes[idx].set_yticks(range(len(top_features)))
        axes[idx].set_yticklabels(top_features.index)
        axes[idx].set_xlabel('Importance')
        axes[idx].set_title(f'{name} Feature Importance')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. End-to-End Pipeline Comparison

In [ ]:
# Compare different feature engineering strategies
strategies = {
    'baseline': {
        'description': 'No feature engineering',
        'transform': lambda X: X
    },
    'statistical': {
        'description': 'Statistical features only',
        'transform': lambda X: feature_engineer._create_statistical_features(
            X.values.mean(axis=1), 'stat'
        )
    },
    'automated': {
        'description': 'Full automated engineering',
        'transform': lambda X: feature_engineer.engineer_features(
            X, include_interactions=True, include_polynomial=True
        )
    }
}

pipeline_results = {}

for strategy_name, strategy in strategies.items():
    print(f"\nTesting {strategy_name}: {strategy['description']}")
    
    try:
        # Apply transformation
        if strategy_name == 'baseline':
            X_train_strategy = X_train
            X_test_strategy = X_test
        else:
            X_train_strategy = strategy['transform'](X_train)
            X_test_strategy = X_test  # Simplified for demo
            
            # Ensure same columns
            common_cols = list(set(X_train_strategy.columns) & set(X_test_strategy.columns))
            X_train_strategy = X_train_strategy[common_cols]
            X_test_strategy = X_test_strategy[common_cols]
        
        # Train model
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_train_strategy, y_train)
        
        # Evaluate
        train_score = model.score(X_train_strategy, y_train)
        test_score = model.score(X_test_strategy, y_test)
        
        pipeline_results[strategy_name] = {
            'train_score': train_score,
            'test_score': test_score,
            'n_features': X_train_strategy.shape[1]
        }
        
        print(f"  Features: {X_train_strategy.shape[1]}")
        print(f"  Train Score: {train_score:.4f}")
        print(f"  Test Score: {test_score:.4f}")
        
    except Exception as e:
        print(f"  Strategy failed: {e}")
        pipeline_results[strategy_name] = {
            'train_score': 0,
            'test_score': 0,
            'n_features': 0
        }

In [ ]:
# Visualize pipeline comparison
if pipeline_results:
    strategies_list = list(pipeline_results.keys())
    train_scores = [pipeline_results[s]['train_score'] for s in strategies_list]
    test_scores = [pipeline_results[s]['test_score'] for s in strategies_list]
    n_features = [pipeline_results[s]['n_features'] for s in strategies_list]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Score comparison
    x = np.arange(len(strategies_list))
    width = 0.35
    
    axes[0].bar(x - width/2, train_scores, width, label='Train', color='skyblue')
    axes[0].bar(x + width/2, test_scores, width, label='Test', color='lightcoral')
    axes[0].set_xlabel('Strategy')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Performance by Strategy')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(strategies_list)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Number of features
    axes[1].bar(strategies_list, n_features, color='lightgreen')
    axes[1].set_xlabel('Strategy')
    axes[1].set_ylabel('Number of Features')
    axes[1].set_title('Feature Count by Strategy')
    axes[1].grid(True, alpha=0.3)
    
    # Efficiency plot
    if any(n > 0 for n in n_features):
        efficiency = [t/n if n > 0 else 0 for t, n in zip(test_scores, n_features)]
        axes[2].bar(strategies_list, efficiency, color='gold')
        axes[2].set_xlabel('Strategy')
        axes[2].set_ylabel('Score per Feature')
        axes[2].set_title('Feature Efficiency')
        axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 8. Target Transformation

In [ ]:
# Test target transformation for regression
X_reg = df_reg.drop('target', axis=1)
y_reg = df_reg['target'].values

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Compare different target transformations
transformations = ['None', 'log', 'sqrt', 'quantile']
transformation_results = {}

for trans in transformations:
    print(f"\nTesting {trans} transformation...")
    
    if trans == 'None':
        y_train_trans = y_train_reg
        y_test_trans = y_test_reg
    else:
        target_transformer = TargetTransformer(method=trans)
        target_transformer.fit(y_train_reg)
        y_train_trans = target_transformer.transform(y_train_reg)
        y_test_trans = y_test_reg  # We'll inverse transform predictions
    
    # Train model
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    model.fit(X_train_reg, y_train_trans)
    
    # Predict
    y_pred_trans = model.predict(X_test_reg)
    
    # Inverse transform if needed
    if trans != 'None':
        y_pred = target_transformer.inverse_transform(y_pred_trans)
    else:
        y_pred = y_pred_trans
    
    # Evaluate
    r2 = r2_score(y_test_reg, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred))
    
    transformation_results[trans] = {
        'r2': r2,
        'rmse': rmse
    }
    
    print(f"  R2: {r2:.4f}, RMSE: {rmse:.4f}")

In [ ]:
# Visualize target transformation effects
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, trans in enumerate(['None', 'log', 'sqrt', 'quantile']):
    if trans == 'None':
        y_plot = y_train_reg[:500]
    else:
        transformer = TargetTransformer(method=trans)
        transformer.fit(y_train_reg)
        y_plot = transformer.transform(y_train_reg[:500])
    
    axes[idx].hist(y_plot, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'{trans.capitalize()} Transformation')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    
    # Add statistics
    axes[idx].axvline(np.mean(y_plot), color='red', linestyle='--', label='Mean')
    axes[idx].axvline(np.median(y_plot), color='green', linestyle='--', label='Median')
    axes[idx].legend()
    
    # Add skewness
    from scipy.stats import skew
    skewness = skew(y_plot)
    axes[idx].text(0.02, 0.98, f'Skew: {skewness:.3f}',
                   transform=axes[idx].transAxes, verticalalignment='top')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated a comprehensive feature engineering pipeline including:

1. **Automated Feature Creation**: Automatic detection of feature types and creation of relevant features
2. **Advanced Transformations**: Outlier handling, skewness correction, dimensionality reduction
3. **Time Series Features**: Extraction of temporal patterns, seasonality, and change points
4. **Feature Selection**: Multiple strategies for selecting the most relevant features
5. **Feature Importance**: Analysis of feature importance across different models
6. **Target Transformation**: Improving model performance through target variable transformation
7. **Pipeline Comparison**: Evaluating different feature engineering strategies

The system provides:
- **Flexibility**: Modular components that can be combined as needed
- **Automation**: Intelligent defaults and automatic feature type detection
- **Comprehensiveness**: Wide range of techniques for different data types
- **Production-ready**: Fit/transform pattern for deployment
- **Interpretability**: Feature importance and selection insights